# YouTube video
## Loading documents from a YouTube url
- https://python.langchain.com/docs/integrations/document_loaders/youtube_audio

### ffprobe or avprobe not found. Please install one
- brew install ffmpeg
- https://stackoverflow.com/questions/30770155/ffprobe-or-avprobe-not-found-please-install-one

In [1]:
import os
import openai
print(openai.__version__)

import sys
sys.path.append('../..')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']


2.21.0


- Use YoutubeAudioLoader to fetch / download the audio files.
- Then, ues OpenAIWhisperParser() to transcribe them to text.
- We will use yt_dlp to download audio for YouTube urls.
- We will use pydub to split downloaded audio files (such that we adhere to Whisper API's 25MB file size limit).

In [2]:
# The new 'community' paths for LangChain 0.1 and later
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import OpenAIWhisperParser
from langchain_community.document_loaders.blob_loaders.youtube_audio import YoutubeAudioLoader

# Note: If you need the OpenAI chat model itself, use:
# from langchain_openai import ChatOpenAI


In [4]:
import yt_dlp
from pathlib import Path
from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.parsers import OpenAIWhisperParser

url = "https://www.youtube.com/watch?v=NMAcu-WS9nE"
save_dir = Path("docs/youtube")
save_dir.mkdir(parents=True, exist_ok=True)

ydl_opts = {
    # 'bestaudio' is usually enough, but we provide fallbacks
    "format": "m4a/bestaudio/best", 
    "outtmpl": str(save_dir / "%(id)s.%(ext)s"),
    "noplaylist": True,
    # Remove the "android" player_client restriction as it's causing the cookie/format error
    "quiet": False, 
    "no_warnings": False,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=True)
    # Use info.get to safely retrieve the extension
    audio_path = Path(ydl.prepare_filename(info))

print(f"Downloaded audio to {audio_path}")


[youtube] Extracting URL: https://www.youtube.com/watch?v=NMAcu-WS9nE
[youtube] NMAcu-WS9nE: Downloading webpage


[youtube] NMAcu-WS9nE: Downloading android vr player API JSON
[info] NMAcu-WS9nE: Downloading 1 format(s): 140
[download] docs/youtube/NMAcu-WS9nE.m4a has already been downloaded
[download] 100% of    4.00MiB
Downloaded audio to docs/youtube/NMAcu-WS9nE.m4a


In [5]:
# Transcribe with Whisper parser
loader = FileSystemBlobLoader(str(save_dir), glob="*.m4a")
parser = OpenAIWhisperParser()

docs = []
for blob in loader.yield_blobs():
    docs = list(parser.lazy_parse(blob))
    break

docs[:1]  # preview first document


Transcribing part 1!


[Document(metadata={'source': 'docs/youtube/NMAcu-WS9nE.m4a', 'chunk': 0}, page_content='每次回家,都覺得門好重。 我不太記得上次跟我爸講話是什麼時候了。 從小,我對爸爸的記憶就是 不是在屋裡看他, 就是看他關門。 爸爸今天學校運動會。 或是,根本沒有他。 或是,他根本沒在乎過我。 這麼晚了,還在處理公司的事。 沒有啦,事關大事而已。 不要想太多啦。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 從一天到一天生活 更渴望 我愛你 希望我是一個無法忽視的人 有時我會覺得 你會說 每天都陪我來 爸爸,父親節快樂! 爸 我愛你')]

In [6]:
len(docs)


1

In [7]:
docs[0]


Document(metadata={'source': 'docs/youtube/NMAcu-WS9nE.m4a', 'chunk': 0}, page_content='每次回家,都覺得門好重。 我不太記得上次跟我爸講話是什麼時候了。 從小,我對爸爸的記憶就是 不是在屋裡看他, 就是看他關門。 爸爸今天學校運動會。 或是,根本沒有他。 或是,他根本沒在乎過我。 這麼晚了,還在處理公司的事。 沒有啦,事關大事而已。 不要想太多啦。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 從一天到一天生活 更渴望 我愛你 希望我是一個無法忽視的人 有時我會覺得 你會說 每天都陪我來 爸爸,父親節快樂! 爸 我愛你')

In [8]:
docs[0].page_content[0:200]


'每次回家,都覺得門好重。 我不太記得上次跟我爸講話是什麼時候了。 從小,我對爸爸的記憶就是 不是在屋裡看他, 就是看他關門。 爸爸今天學校運動會。 或是,根本沒有他。 或是,他根本沒在乎過我。 這麼晚了,還在處理公司的事。 沒有啦,事關大事而已。 不要想太多啦。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 '

## Save docs to a text file

In [9]:
from pathlib import Path

# pick a filename
out_file = Path("docs/youtube/transcript.txt")

with out_file.open("w", encoding="utf-8") as f:
    for i, d in enumerate(docs, 1):
        f.write(f"--- Document {i} ---\n")
        f.write(d.page_content.strip() + "\n\n")

print(f"Transcript saved to {out_file}")


Transcript saved to docs/youtube/transcript.txt


## Load it back later

In [3]:
from pathlib import Path

in_file = Path("docs/youtube/transcript.txt")

with in_file.open("r", encoding="utf-8") as f:
    transcript_text = f.read()

# The updated path for LangChain 0.1+
from langchain_core.documents import Document

# Your existing logic
docs = [Document(page_content=transcript_text)]

print(docs[0].page_content[:500], "...")

--- Document 1 ---
每次回家,都覺得門好重。 我不太記得上次跟我爸講話是什麼時候了。 從小,我對爸爸的記憶就是 不是在屋裡看他, 就是看他關門。 爸爸今天學校運動會。 或是,根本沒有他。 或是,他根本沒在乎過我。 這麼晚了,還在處理公司的事。 沒有啦,事關大事而已。 不要想太多啦。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 小雨啊,去爸爸的房間幫他擦乾淨。 從一天到一天生活 更渴望 我愛你 希望我是一個無法忽視的人 有時我會覺得 你會說 每天都陪我來 爸爸,父親節快樂! 爸 我愛你

 ...


## Building a chat app from YouTube video

In [4]:
import langchain
import langchain_core
print(f"LangChain: {langchain.__version__}")
print(f"Core: {langchain_core.__version__}")


LangChain: 0.3.27
Core: 0.3.83


In [5]:
# From the main 'langchain' package
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter

# From 'langchain_community'
from langchain_community.vectorstores import FAISS

# From 'langchain_openai'
from langchain_openai import ChatOpenAI

In [6]:
# Combine doc
combined_docs = [doc.page_content for doc in docs]
text = " ".join(combined_docs)


In [7]:
# Split them
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
splits = text_splitter.split_text(text)


In [9]:
# The new import path for LangChain 0.1+
from langchain_openai import OpenAIEmbeddings

# Build an index
embeddings = OpenAIEmbeddings()
vectordb = FAISS.from_texts(splits, embeddings)


In [23]:
# Build a QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0),
    chain_type="stuff",
    retriever=vectordb.as_retriever(),
)


In [24]:
# Ask a question!
query = "我對爸爸的記憶是什麼?"

# The old way:
# qa_chain.run(query)

# The new, supported way:
response = qa_chain.invoke({"query": query})

# Note: .invoke() returns a dictionary. 
# To get just the text answer, you access the 'result' key:
print(response["result"])


你對爸爸的記憶主要是他在屋裡的樣子，或者是看他關門的情景。你提到從小就有這樣的印象，並且感受到他似乎不太在乎你。


In [25]:
query = "什麼時候學校運動會?"
response = qa_chain.invoke({"query": query})
print(response["result"])


我不知道。


In [26]:
query = "爸爸有去學校運動會嗎?"
response = qa_chain.invoke({"query": query})
print(response["result"])


是的，爸爸今天去學校運動會。


## Summarizing

In [20]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def get_completion(prompt, model="gpt-4o-mini"):
    try:
        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0, # this is the degree of randomness of the model's output
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"⚠️ Error: {str(e)}"
    

### Summarize with a word/sentence/character limit

In [21]:
prompt = f"""
你的任務是產生一個劇情或故事的簡短摘要。

將下方被三個反引號標記的故事摘要化，最多60個字。

故事： ```{docs[0].page_content}```
"""

response = get_completion(prompt)
print(response)


一名女孩對父親的疏離感到失落，回憶起童年時光，渴望父愛與關注，並在父親節表達心聲。


## Summarize with a focus ...

In [38]:
prompt = f"""
你的任務是產生一個劇情或故事的簡短摘要。

將下方被三個反引號標記的故事摘要化，並專注於感人或揪心的情節或情境。限制在60個字以內。

故事： ```{docs[0].page_content}```
"""

response = get_completion(prompt)
print(response)


一個孤獨的孩子，對父親的渴望和失望交織。父親總是忙碌，孩子卻渴望被關注和愛。父親節來臨，孩子期待著一句祝福，卻只有空虛和失望。他們彼此間的距離和孤獨，讓人心碎。


## Try "extract" instead of "summarize"

In [22]:
prompt = f"""
你的任務是產生一個劇情或故事的簡短摘要。

將下方被三個反引號標記的故事中，提取與感人或揪心的情節或情境。限制在60個字以內。

故事： ```{docs[0].page_content}```
"""

response = get_completion(prompt)
print(response)


小雨渴望父親的關注，卻總是感受到距離與冷漠。每次回家，重重的門象徵著他們之間的隔閡，心中默默呼喚著「我愛你」。
